In [11]:
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry, ModelType
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from utils.graph_builder import LocalizationGraphBuilder, traverse_namespaces
from adaptation.misc import NameAnonymizer
from schemas.similarity import SearchMethod
import os


In [12]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} spacy={'es': 'es_core_news_sm'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/vectors.bin'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/bilstm_attention_cosine'} models=[<SearchMethod.SBERT: 'sbert'>, <SearchMethod.WORD2VEC_IDF: 'word2vec_idf'>] allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000 spell_max_distance=2 spell_unigram_weight=0.8 spell_data_dir='./spelling_checker/data' spell_model_dir='./spelling_checker/models' faiss_data_dir='./faiss_data' adaptation_data_dir='./adaptation/data' localization_dir='./adaptation/localization'


In [13]:
adaptation_dir = "./adaptation"

localization_dir = os.path.join(adaptation_dir, "localization")
language_dir = os.path.join(localization_dir, "dialogue", "active")
structure_dir = os.path.join(localization_dir,  "structure", "modified")

database_dir = "./faiss_data"

data_dir = os.path.join(adaptation_dir, "data")
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [14]:
namespaces = traverse_namespaces(language_dir, languages)
print(namespaces)


['scene5/scene5Bedroom', 'scene4/scene4Garage', 'scene6/routeB/scene6BedroomRouteB', 'scene1/scene1Bedroom2', 'computer/captions', 'generalDialogs', 'scene6/routeB/scene6PoliceStationRouteB', 'scene1/scene1Lunch1', 'menus/titleScene', 'scene1/scene1Bedroom1', 'dialogManager', 'transitions', 'scene2/scene2Break', 'menus/creditsScene', 'computer/usernames', 'scene5/scene5Livingroom', 'scene4/scene4Frontyard', 'scene2/scene2Bedroom', 'scene6/routeA/scene6LunchRouteA', 'scene4/scene4Bedroom', 'scene7/scene7Bedroom', 'scene6/routeA/scene6PortalRouteA', 'scene3/scene3Break', 'scene1/scene1Classroom', 'scene6/routeB/scene6LunchRouteB', 'computer/socialMediaScreen', 'scene6/routeB/scene6EndingRouteB', 'scene6/routeA/scene6BedroomRouteA2', 'computer/loginScreen', 'scene3/scene3Bedroom', 'scene1/scene1Break', 'deviceInfo', 'scene6/routeA/scene6EndingRouteA', 'scene4/scene4Backyard', 'scene6/scene6Livingroom', 'scene6/routeA/scene6BedroomRouteA1', 'menus/loginScene', 'scene6/scene6Bedroom', 'scen

In [15]:
backend = Backend(
    name_mapping=lambda lng, ns: os.path.join(
        language_dir,
        lng,
        f"{ns}.json"
    )
)

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [16]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="[UNK]"
)


In [17]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer(ModelType.SBERT)
model_registry.build_lstm()
# model_registry.build_spacy()
# model_registry.build_word2vec()
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)


2026-07-21 00:12:20.696 | DEBUG    | services.model_registry:_create_loader:59 - Registering sbert loader for 'es'.
2026-07-21 00:12:20.696 | DEBUG    | services.model_registry:_create_loader:59 - Registering lstm loader for 'es'.
2026-07-21 00:12:20.700 | DEBUG    | services.model_registry:_create_loader:59 - Registering lstm calibrator loader for 'es'.
2026-07-21 00:12:20.701 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-07-21 00:12:22.136 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'
2026-07-21 00:12:22.137 | DEBUG    | services.lazy_loader:model:16 - Loading lstm for 'es'...


Using device: cuda


2026-07-21 00:12:24.426 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded lstm for 'es'
2026-07-21 00:12:24.427 | DEBUG    | services.lazy_loader:model:16 - Loading lstm calibrator for 'es'...
2026-07-21 00:12:24.470 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded lstm calibrator for 'es'


In [18]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir=structure_dir,
    model_types=[
        SearchMethod.SBERT,
        SearchMethod.LSTM
	]
)

builder.run()


2026-07-21 00:12:24.575 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 33 vectors


lstm


2026-07-21 00:12:24.929 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 33 vectors
2026-07-21 00:12:25.016 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-21 00:12:25.167 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-21 00:12:25.229 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 31 vectors
2026-07-21 00:12:25.383 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 31 vectors
2026-07-21 00:12:25.493 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 48 vectors
2026-07-21 00:12:25.618 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 48 vectors
2026-07-21 00:12:25.679 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-21 00:12:25.820 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-21 00:12:25.906 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 35 vectors
2026-07-21 00:12:26.063 | DEBUG    | controllers.r

Total visited nodes: 732


In [19]:
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
test_engine = multilingual.get_node_engine("es", SearchMethod.SBERT)

print(test_engine.retrievers)

test_engine.load_all()

# test_engine.load_node("scene1Bedroom1_computer1_choices_similarity")

print(test_engine.retrievers)


2026-07-21 00:12:27.412 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Bedroom1_computer1_choices_similarity
2026-07-21 00:12:27.413 | SUCCESS  | services.node_engine:load_node:88 - Loaded node successfully.
2026-07-21 00:12:27.414 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Bedroom1_computer2_root
2026-07-21 00:12:27.416 | SUCCESS  | services.node_engine:load_node:88 - Loaded node successfully.
2026-07-21 00:12:27.416 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Bedroom2_computer_choices2_similarity
2026-07-21 00:12:27.417 | SUCCESS  | services.node_engine:load_node:88 - Loaded node successfully.
2026-07-21 00:12:27.418 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Classroom_part2_thanks_similarity
2026-07-21 00:12:27.418 | SUCCESS

{}
{'scene1Bedroom1_computer1_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC91616780>, 'scene1Bedroom1_computer2_root': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC9165AC60>, 'scene1Bedroom2_computer_choices2_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC9165ADB0>, 'scene1Classroom_part2_thanks_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC6ED24D70>, 'scene2Break_part2_choice_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC6ED21310>, 'scene3Bedroom_main_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC6ED20110>, 'scene4Backyard_mainConversation_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC6ECF1010>, 'scene4Bedroom_phone_root': <controllers.retrievers.faiss.FaissRetriever object at 0x000001DC6ECF20F0>, 'scene4Garage_phone1_root': <controllers.retrievers.fais

In [20]:
retriever = test_engine.get_retriever("scene1Bedroom1_computer1_choices_similarity")

retriever.search("Hola", 3)


(array([10, 12, 11], dtype=int32),
 array([0.5469646 , 0.53729975, 0.48407146], dtype=float32),
 array(['Hola. En serio?', 'Holaa, pues bien', 'Hola jaja. Supongo'],
       dtype=object))